# Carbon-Aware Scheduling with CA-WOA — Revision Artefact

**Navaneetha Thalakokkula — MSc Cloud Computing, National College of Ireland**

This notebook reproduces the revised results. It differs from the submitted notebook
in five ways, each of which fixes a real defect:

| # | Defect in the submitted notebook | Fix here |
|---|---|---|
| 1 | `mealpy` installed **unpinned** — Colab would install whatever version is current, not the 3.0.3 reported in the paper | pinned to `mealpy==3.0.3` |
| 2 | Carbon intensity, the 60 tasks and the workload series were **hardcoded arrays** | loaded from the real traces in `data/` |
| 3 | The 60 sampled tasks all shared one submit timestamp, so **every arrival collapsed to slot 0** and the workload occupied 4 of 144 slots | even-stride sampling across all 52,057 usable tasks, preserving real submit ordering |
| 4 | `GA.OriginalGA` performs **no search** in mealpy 3.0.3 (40 objective evaluations instead of 4,840) | `GA.BaseGA`; the defect is measured explicitly rather than silently fixed |
| 5 | The scalability table was a **hardcoded `SWEEP` dict**, stale by up to 5.4 pp versus the computed CSVs | every table is computed, or read from committed per-seed raw CSVs |

**Runtime note.** The full experiment matrix is ~40,000 runs and takes roughly 2.5 h
on 12 CPU cores. Free Colab provides about 2, so this notebook *verifies* the
pipeline on a reduced configuration and then renders the published tables from the
committed raw CSVs. To regenerate everything from scratch, run `revision/runner.py`
offline.

## 1. Environment — pinned, reproducible

In [ ]:
!pip -q install "mealpy==3.0.3" "numpy==1.26.0" "scipy>=1.11" pandas matplotlib
!git clone -q https://github.com/<YOUR-USERNAME>/carbon-aware-scheduler.git 2>/dev/null || echo "already cloned"
%cd carbon-aware-scheduler/revision

import mealpy, numpy, scipy, pandas
print("mealpy", mealpy.__version__, "| numpy", numpy.__version__,
      "| scipy", scipy.__version__, "| pandas", pandas.__version__)
assert mealpy.__version__ == "3.0.3", "version pin failed - results will not match the paper"

## 2. Validation gate — run this before trusting anything below

`validate.py` contains 27 checks in five groups. The two that matter most:

- **Group A** proves the fast vectorised `evaluate()` is numerically identical to the
  original dict-based implementation from `src/week4_full_comparison.py` — worst
  relative deviation `9.3e-16`, i.e. machine precision.
- **Group B** rebuilds the *published* setup and reproduces the submitted Table 4
  exactly: consolidation 80.50, carbon-aware greedy 82.56, greedy utilisation 75.2,
  naive FIFO energy 5.033 kWh, derived M=5.

If any check fails, nothing downstream is trustworthy.

In [ ]:
!python validate.py

## 3. The corrected workload

The submitted notebook drew its 60 tasks with `head(n)`, which returned tasks sharing
a single submit timestamp. Every arrival therefore collapsed to slot 0 and the whole
workload occupied 4 of the 144 available slots — so there was almost nothing for a
temporal-shifting scheduler to shift. Below, both samplings are built side by side so
the difference is visible rather than asserted.

In [ ]:
import numpy as np, pandas as pd, core as K

CI, PRICE, H = K.load_carbon()
print("carbon slots:", H, "| CI range %.0f-%.0f gCO2/kWh" % (CI.min(), CI.max()))

pub = K.build_published(60)                 # as submitted: head(60)
fix = K.build_google(60, H)                 # corrected: even stride
for name, t in (("submitted (head)", pub), ("corrected (stride)", fix)):
    e = np.asarray(t["earliest"])
    print("%-20s distinct arrival slots: %3d   span: %d slots"
          % (name, len(np.unique(e)), e.max() - e.min() + 1))

## 4. Reduced live reproduction

A single configuration run end-to-end so you can see the pipeline actually execute.
N=500 with 5 seeds takes about a minute on Colab; the published tables below use
N=500-3000 with 30 seeds.

In [ ]:
from mealpy import FloatVar, WOA
import time

env = K.make_env(n=500, M=10, hard=True, pen=10.0)
print("tasks %d | hosts %d | peak demand %.1f | horizon %d slots"
      % (env.N, env.M, env.peak, env.H))

rows = []
for name, starts in (("FIFO (naive)", env.fifo),
                     ("Greedy EDF+greenest", env.greedy_starts())):
    m = env.evaluate(np.asarray(starts, int), consolidate=True)
    rows.append((name, env.cred(m), m["SLA_%"], m["Overload_%"], m["Feasible"]))

for seed in range(1, 6):
    rng = np.random.default_rng(seed)
    env.nfe = 0
    res = WOA.OriginalWOA(epoch=120, pop_size=40).solve(
        {"obj_func": env.fitness,
         "bounds": FloatVar(lb=[0.0]*env.N, ub=[1.0]*env.N),
         "minmax": "min", "log_to": None},
        starting_solutions=env.pop_carbon(40, rng), seed=seed)
    m = env.evaluate(env.decode(res.solution), consolidate=True)
    rows.append(("CA-WOA seed %d" % seed, env.cred(m), m["SLA_%"],
                 m["Overload_%"], m["Feasible"]))

pd.DataFrame(rows, columns=["method", "carbon_reduction_%", "SLA_%",
                            "overload_%", "feasible"]).round(3)

## 5. Published results — from the committed per-seed CSVs

Each `raw_*.csv` holds **one row per (configuration, method, seed)**: the per-seed
results the reviewers asked for, not summary statistics. Nothing below is transcribed
by hand.

In [ ]:
import glob, os
label = lambda r: "CA-WOA" if (r["method"] == "WOA" and r["init"] == "carbon") else r["method"]

def load(exp):
    d = pd.read_csv("raw_%s.csv" % exp)
    d["label"] = d.apply(label, axis=1)
    return d

print("available raw result files:")
for f in sorted(glob.glob("raw_*.csv")):
    print("   %-24s %6d rows" % (f, len(pd.read_csv(f))))

### 5.1 Capacity as a feasibility constraint (E20)

This is the experiment that decides the paper's framing. Host capacity is treated as
a **feasibility constraint**: energy is charged on the full demanded load, so
overloading is never rewarded, and a schedule counts as feasible only when overload
is exactly zero.

The heuristic baselines reach their headline carbon numbers by overloading hosts —
greedy overloads up to 45% at 3000 tasks on 10 hosts. Its 89.40% is therefore an
*infeasible upper bound*, not a competing result.

In [ ]:
d = load("E20")
print("Feasibility rate (fraction of 30 seeds with zero overload):")
display(d.pivot_table(index="label", columns=["N", "M"], values="feasible").round(2))

print("\nOverload % of demanded load — the heuristics break down as load rises:")
display(d[d.label.isin(["FIFO", "Consolidation", "Greedy(EDF+greenest)"])]
        .pivot_table(index="label", columns=["N", "M"], values="overload_%").round(2))

print("\nCarbon reduction among FEASIBLE runs, mean +/- sd:")
f = d[d.feasible]
display(f.groupby(["N", "M", "label"])["carbon_red_vs_naive_%"]
        .agg(["mean", "std"]).round(2).unstack(0))

### 5.2 Ablation — the carbon-aware seeding is the contribution

Two results the paper must state plainly:

- **Carbon-aware seeding works, and generalises beyond WOA** — it helps in 59 of 60
  (algorithm, N, M) cells across both energy models.
- **"Improved" initialisation without carbon data is worth nothing.** Latin-hypercube
  plus opposition-based sampling is statistically indistinguishable from uniform
  random (p = 0.06-0.94). The gain comes from the carbon information, not from a
  more sophisticated sampling scheme.

In [ ]:
from scipy import stats

def seeding_effect(exp):
    d = load(exp); d = d[(d.nfe > 0) & (d.beta > 0) & (d.gamma > 0)]
    out = []
    for (N, M, algo), g in d.groupby(["N", "M", "method"]):
        r = g[g.init == "random"]["carbon_red_vs_naive_%"]
        c = g[g.init == "carbon"]["carbon_red_vs_naive_%"]
        if len(r) < 5 or len(c) < 5:
            continue
        p1 = stats.ttest_ind(c, r, equal_var=False).pvalue
        p2 = stats.mannwhitneyu(c, r).pvalue
        out.append({"N": N, "M": M, "algo": algo, "gain_pp": c.mean() - r.mean(),
                    "significant": bool(p1 < 0.05 and p2 < 0.05)})
    return pd.DataFrame(out)

for exp, tag in (("E3", "published model"), ("E22", "capacity constraint")):
    t = seeding_effect(exp)
    print("%-22s helps in %d of %d cells | mean %+.2f pp | significant in %d"
          % (tag, (t.gain_pp > 0).sum(), len(t), t.gain_pp.mean(), t.significant.sum()))

### 5.3 Fitness weights — what each term actually buys

Reading the carbon column alone is misleading: dropping the SLA term (beta) or the
overload term (gamma) *raises* carbon reduction, but only by breaking the constraint
that term exists to protect. Setting alpha=1.0 reaches the best carbon figure in the
study while violating 71% of deadlines.

The published weights (0.4, 0.3, 0.3) sit on the knee of that trade-off.

In [ ]:
d = load("E5")
d["abg"] = d.apply(lambda r: "(%.2f,%.2f,%.2f)" % (r.alpha, r.beta, r.gamma), axis=1)
w = d[(d.init == "carbon") & (~d.cap) & (d.seed_frac.round(4) == 0.3333)]
piv = w.groupby("abg")[["carbon_red_vs_naive_%", "sla_%", "overload_%"]].mean().round(2)
piv.columns = ["carbon_%", "SLA_%", "overload_%"]
display(piv.sort_values("carbon_%", ascending=False))
print("Highest-carbon settings are the ones that break deadlines or overload hosts.")

### 5.4 Power models — the conclusion does not depend on the linear model

Reviewers asked whether the published linear power model flatters the method. It does
not: CA-WOA ranks first in all 6 cells under each of linear, cubic (convex, u^3) and
piecewise (SPECpower-style) models — 18 of 18. Its margin is in fact *widest* under
the cubic model.

In [ ]:
d = load("E2")
display(d[d.nfe > 0].pivot_table(index="label", columns=["power", "N"],
                                 values="carbon_red_vs_naive_%").round(2))

### 5.5 Computational cost — the equal-budget claim was wrong

Section 6.1 of the submitted paper states that all algorithms receive an equal
evaluation budget. They do not: HHO takes roughly 8,900 objective evaluations against
4,840 for every other method. `GA.OriginalGA` takes 40.

In [ ]:
a = pd.concat([load(e) for e in ("E1", "E2", "E3", "E4", "E20", "E21", "E22")],
              ignore_index=True)
a = a[a.nfe > 0]
print("Objective-function evaluations per run:")
display(a.groupby("method")["nfe"].agg(["mean", "min", "max"]).round(0))
print("\nRuntime (s) and peak memory (MB) by task count:")
display(a.pivot_table(index="method", columns="N",
                      values=["runtime_s", "peak_mem_mb"], aggfunc="mean").round(2))

g = load("GABUG")
print("\nThe GA.OriginalGA defect, measured rather than assumed:")
display(g.groupby(["method", "N"])[["nfe", "carbon_red_vs_naive_%", "runtime_s"]]
        .mean().round(2))

### 5.6 A second workload — NASA-iPSC real arrival process

The Google trace covers only 1.54 h of arrivals, so arrival times must be rescaled
onto the horizon. The NASA-iPSC trace supplies a genuine 92-day arrival process as an
independent check. Carbon reductions are lower there (60-72% against 84-89%) because
a real multi-day arrival spread offers less shifting opportunity — and the paper's
central SLA claim shows up far more strongly.

In [ ]:
d = load("E7"); n = d[(d.wl == "nasa") & (d.nfe > 0)]
display(n.groupby(["N", "M", "label"])[["carbon_red_vs_naive_%", "sla_%"]]
        .mean().round(2).unstack(0))
print("PSO/DE/GA miss 38-64% of deadlines; CA-WOA stays below 0.5%.")

### 5.7 Minimum active-host constraint — and how it reverses the comparison

A datacentre cannot scale to zero: a warm pool stays powered across the operating
window and draws idle power even in slots with no work. That makes deferral costly,
because shifting a task into a greener slot extends the window and keeps the pool
running longer.

This single realistic constraint **reverses the headline comparison**. Without it the
greedy heuristic beats CA-WOA two-thirds of the time; with it, CA-WOA wins every
configuration.

The same experiment also contains the most important caveat in the revision, and it
is reported here rather than buried: in the regime where the metaheuristic finally
beats greedy, the carbon-aware *seeding* contributes almost nothing. The win comes
from the WOA search itself.

In [ ]:
from scipy import stats
d = load("E23")

rows = []
for mm in ["0", "0.25", "0.5", "auto"]:
    s = d[d.mmin.astype(str) == mm]
    vs_greedy, vs_woa, sig, n = [], [], 0, 0
    for (N, M), g in s.groupby(["N", "M"]):
        ca = g[g.label == "CA-WOA"]["carbon_red_vs_naive_%"]
        gr = g[g.label == "Greedy(EDF+greenest)"]["carbon_red_vs_naive_%"]
        wo = g[g.label == "WOA"]["carbon_red_vs_naive_%"]
        if len(gr):
            vs_greedy.append(ca.mean() - gr.mean())
        if len(wo) > 3:
            n += 1
            vs_woa.append(ca.mean() - wo.mean())
            sig += (stats.ttest_ind(ca, wo, equal_var=False).pvalue < 0.05
                    and stats.mannwhitneyu(ca, wo).pvalue < 0.05)
    rows.append({"min_active_hosts": mm,
                 "beats_greedy": "%d/%d" % (sum(1 for x in vs_greedy if x > 0), len(vs_greedy)),
                 "vs_greedy_pp": round(np.mean(vs_greedy), 2),
                 "beats_plain_WOA": "%d/%d" % (sum(1 for x in vs_woa if x > 0), n),
                 "vs_WOA_pp": round(np.mean(vs_woa), 2),
                 "WOA_diff_significant_in": "%d/%d" % (sig, n)})
display(pd.DataFrame(rows))

### 5.8 A modern published scheduler as baseline

Reviewer 3 asked for comparison against a modern scheduler, not only heuristics.
Two published policies are reimplemented in the same simulator, scored by the same
metrics, using no fitness function and no metaheuristic machinery:

- **VCC** — the virtual-capacity-curve load-shaping policy of Google's production
  carbon-aware scheduler (Radovanovic et al., *IEEE Trans. Power Systems* 38(2),
  2023). Per-slot capacity is throttled as a function of carbon intensity.
- **Threshold** — temporal shifting from Wiesner et al., *Let's Wait Awhile*
  (ACM/IFIP Middleware 2021): defer while intensity sits above a percentile.

VCC's throttle floor was swept over 8 values rather than taking the first guess, so
the baseline is not accidentally crippled; it proved insensitive (85.02-85.11 at
N=1000). Only VCC and CA-WOA are feasible in every cell.

In [ ]:
d = load("E24"); d = d[d.hard]
p = d.pivot_table(index=["N", "M"], columns="label", values="carbon_red_vs_naive_%")
o = d.pivot_table(index=["N", "M"], columns="label", values="overload_%")
print("Feasible in all 12 cells:",
      [c for c in o.columns if (o[c] == 0).all()])
p["CA-WOA - VCC"] = (p["CA-WOA"] - p["VCC(Google)"]).round(2)
display(p[["CA-WOA", "VCC(Google)", "Greedy(EDF+greenest)", "CA-WOA - VCC"]].round(2))
print("CA-WOA beats Google's VCC policy in %d of %d cells."
      % ((p["CA-WOA - VCC"] > 0).sum(), len(p)))
print("Greedy's higher numbers are infeasible - it overloads hosts by up to %.0f%%."
      % o["Greedy(EDF+greenest)"].max())

### 5.9 Convergence behaviour

Reviewers asked for convergence curves and total objective-function evaluations.
Both are recorded per run. The seeding shows up exactly as claimed: CA-WOA starts at
roughly half plain WOA's fitness and stabilises in 8 epochs against 16.

**Caveat reported rather than hidden:** GA and GWO were still improving at epoch 120,
so the fixed budget under-serves them and their results are a lower bound.

In [ ]:
d = load("CONV"); d = d[d.curve.notna()]
s = d[(d.N == 3000) & (d.M == 10)]
rows = []
for L, g in s.groupby("label"):
    m = np.array([[float(x) for x in c.split(";")] for c in g.curve]).mean(0)
    rows.append({"method": L, "epoch_1": round(m[0], 5), "epoch_10": round(m[9], 5),
                 "epoch_120": round(m[119], 5),
                 "improvement_%": round(100 * (m[0] - m[119]) / m[0], 1),
                 "epochs_to_within_1%": int(np.argmax(m <= m[-1] * 1.01)) + 1})
display(pd.DataFrame(rows).sort_values("epoch_120"))

### 5.10 Multiple carbon regions

The submitted work scheduled against a single 3-day UK window. Reviewers asked for
additional regions and for evidence that the chosen signal is representative.

Four real half-hourly traces are now used, spanning the same 111-day period. Ireland
and Northern Ireland come from the EirGrid Smart Grid Dashboard (no API key);
California is derived from the EIA's hourly fuel mix using IPCC AR5 median lifecycle
emission factors, since the EIA does not publish carbon intensity directly.

Two limitations are stated rather than smoothed over: the EIA publishes hourly, so
the US series is upsampled to 30-minute slots, and the UK time-of-use tariff is
applied unchanged to every region as a cost proxy.

In [ ]:
import core as K
rows = []
for r in ("UK", "IE", "NI", "US-CAL", "US-MIDA"):
    try:
        w = K.carbon_windows(r)
    except Exception:
        continue
    allv = np.concatenate(w)
    rows.append({"region": r, "disjoint_3day_windows": len(w),
                 "mean_gCO2_kWh": round(allv.mean(), 1),
                 "min": round(allv.min()), "max": round(allv.max()),
                 "within_window_spread": round(np.mean([x.max() - x.min() for x in w]), 1)})
display(pd.DataFrame(rows))

for exp, title in (("E26", "carbon reduction by region"),
                   ("E27", "all 37 UK windows - is the published one representative?")):
    try:
        d = load(exp)
    except FileNotFoundError:
        print("%s not present - run `python runner.py %s`" % (exp, exp)); continue
    print("\n" + title)
    idx = "region" if exp == "E26" else "window"
    display(d[d.label.isin(["CA-WOA", "Greedy(EDF+greenest)", "VCC(Google)"])]
            .pivot_table(index=idx, columns="label",
                         values="carbon_red_vs_naive_%").round(2))

## 6. Summary

1. **Carbon-aware seeding is the contribution**, and it is not WOA-specific — it
   helps in 59 of 60 (algorithm, configuration) cells.
2. **Initialisation without carbon data buys nothing**, which isolates *what* the
   seeding contributes.
3. **Under host-capacity limits the greedy heuristic becomes infeasible** while
   population-based search stays feasible; that is the regime where the metaheuristic
   earns its place. Without capacity limits the objective is separable and the greedy
   rule is provably optimal — see `NOTES_constrained_baseline.md`.
4. **The conclusion survives the power model**, holding under linear, cubic and
   piecewise models alike.
5. **Reported honestly:** CA-WOA loses to GA in 3 of 12 capacity-constrained cells,
   all at M=10, by 0.08-0.15 pp; HHO received an unequal evaluation budget; and the
   submitted paper's GA baseline performed no search.